In [5]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import geopandas as gpd
import pydeck as pdk

file_path = "data/processed/tracts_with_predictions.parquet"

print("=" * 70)
print("Interactive Map with Proper Color Ramp (PyDeck)")
print("=" * 70)

# ============================================================
# 1. LOAD DATA
# ============================================================

gdf = gpd.read_parquet(file_path).to_crs(epsg=4326)

# Create percentage column for tooltip
gdf["predicted_prob_pct"] = (gdf["predicted_prob"] * 100).round(1)
gdf["predicted_prob"] = gdf["predicted_prob"].fillna(0)

# ============================================================
# 2. DEFINE LAYER WITH BETTER COLOR RAMP
# ============================================================

layer = pdk.Layer(
    "GeoJsonLayer",
    gdf,
    opacity=0.85,
    stroked=True,
    filled=True,
    get_fill_color=[
        # Red channel: increases sharply after ~0.5
        "predicted_prob < 0.5 ? 34 + (110 * predicted_prob * 2) : 255",
        # Green channel: high in the middle, drops at both ends
        "predicted_prob < 0.5 ? 139 + (99 * predicted_prob * 2) : 255 * (1 - (predicted_prob - 0.5) * 2)",
        # Blue channel: low throughout
        "50 * (1 - predicted_prob)"
    ],
    get_line_color=[255, 255, 255],
    get_line_width=0.3,
    pickable=True,
    auto_highlight=True,
)

# ============================================================
# 3. TOOLTIP (Percentage)
# ============================================================

tooltip = {
    "html": """
    <b>Predicted Probability:</b> {predicted_prob_pct}%<br/>
    <b>Actual Wind Farm:</b> {has_wind_farm}
    """,
    "style": {
        "backgroundColor": "#2E86AB",
        "color": "white",
        "fontSize": "13px"
    }
}

# ============================================================
# 4. CREATE THE DECK
# ============================================================

view_state = pdk.ViewState(
    latitude=35.0,
    longitude=-98.0,
    zoom=5,
    pitch=0,
    bearing=0
)

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip=tooltip,
    map_style="mapbox://styles/mapbox/light-v9"
)

deck.show()

# ============================================================
# 5. SAVE AS HTML
# ============================================================

deck.to_html("figures/wind_farm_predicted_probability_map.html")
print("\nInteractive map saved to: figures/wind_farm_predicted_probability_map.html")

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Interactive Map with Proper Color Ramp (PyDeck)

Interactive map saved to: figures/wind_farm_predicted_probability_map.html
